<a href="https://colab.research.google.com/github/Eddydev-ALU/Malaria_Diagnosis_Models/blob/main/Malaria_Diagnosis_CNN_Group16_Transfer_learning_mobile_netv2__Kalisa_ivan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning for Malaria Diagnosis
This notebook is inspired by works of (Sivaramakrishnan Rajaraman  et al., 2018) and (Jason Brownlee, 2019). Acknowledge to NIH and Bangalor Hospital who make available this malaria dataset.

Malaria is an infectuous disease caused by parasites that are transmitted to people through the bites of infected female Anopheles mosquitoes.

The Malaria burden with some key figures:
<font color='red'>
* More than 219 million cases
* Over 430 000 deaths in 2017 (Mostly: children & pregnants)
* 80% in 15 countries of Africa & India
  </font>

![MalariaBurd](https://github.com/habiboulaye/ai-labs/blob/master/malaria-diagnosis/doc-images/MalariaBurden.png?raw=1)

The malaria diagnosis is performed using blood test:
* Collect patient blood smear
* Microscopic visualisation of the parasit

![MalariaDiag](https://github.com/habiboulaye/ai-labs/blob/master/malaria-diagnosis/doc-images/MalariaDiag.png?raw=1)
  
Main issues related to traditional diagnosis:
<font color='#ed7d31'>
* resource-constrained regions
* time needed and delays
* diagnosis accuracy and cost
</font>

The objective of this notebook is to apply modern deep learning techniques to perform medical image analysis for malaria diagnosis.

*This notebook is inspired by works of (Sivaramakrishnan Rajaraman  et al., 2018), (Adrian Rosebrock, 2018) and (Jason Brownlee, 2019)*

## Configuration

In [ ]:
#Mount the local drive project_forder
from google.colab import drive
drive.mount('/content/drive/')
!ls "/content/drive/My Drive/Colab Notebooks/10xDS/Projects/malaria-diagnosis/"

In [ ]:
# Use GPU: Please check if the outpout is '/device:GPU:0'
import tensorflow as tf
print(tf.__version__)
tf.test.gpu_device_name()
#from tensorflow.python.client import device_lib
#device_lib.list_local_devices()

## Populating namespaces

In [ ]:
# Importing basic libraries
import os
import random
import shutil
from matplotlib import pyplot
from matplotlib.image import imread
%matplotlib inline

# Importing the Keras libraries and packages
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Convolution2D as Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dense

In [ ]:
# Define the useful paths for data accessibility
ai_project = '.' #"/content/drive/My Drive/Colab Notebooks/ai-labs/malaria-diagnosis"
cell_images_dir = os.path.join(ai_project,'cell_images')
training_path = os.path.join(ai_project,'train')
testing_path = os.path.join(ai_project,'test')

## Prepare DataSet

### *Download* DataSet

In [ ]:
# Download the data in the allocated google cloud-server. If already down, turn downloadData=False
downloadData = True
if downloadData == True:
  indrive = False
  if indrive == True:
    !wget https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip -P "/content/drive/My Drive/Colab Notebooks/ai-labs/malaria-diagnosis"
    !unzip "/content/drive/My Drive/Colab Notebooks/ai-labs/malaria-diagnosis/cell_images.zip" -d "/content/drive/My Drive/Colab Notebooks/ai-labs/malaria-diagnosis/"
    !ls "/content/drive/My Drive/Colab Notebooks/ai-labs/malaria-diagnosis"
  else: #incloud google server
    !rm -rf cell_images.*
    !wget https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip
    !unzip cell_images.zip >/dev/null 2>&1
    !ls

## Baseline CNN Model
Define a basic ConvNet defined with ConvLayer: Conv2D => MaxPooling2D followed by Flatten => Dense => Dense(output)

![ConvNet](https://github.com/habiboulaye/ai-labs/blob/master/malaria-diagnosis/doc-images/ConvNet.png?raw=1)


## Transfer Learning — MobileNetV2 Fine-Tuning
In this section, we build a transfer learning model using the MobileNetV2 architecture, pre-trained on ImageNet. We will fine-tune the model to classify cell images as parasitized or uninfected.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns

# Hyperparameters
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 10 # Adjust for more thorough experiments
LEARNING_RATE = 1e-4

# Build the MobileNetV2 Model
def build_mobilenet_model(input_shape=(128, 128, 3)):
    # Load base model, excluding top fully connected layers
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape)

    # Freeze the base model initially
    base_model.trainable = False

    # Add custom fully connected layers on top
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    predictions = Dense(1, activation='sigmoid')(x) # Binary classification

    model = Model(inputs=base_model.input, outputs=predictions)

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                  loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')])
    return model, base_model

mobilenet_model, base_model = build_mobilenet_model()
mobilenet_model.summary()

### Data Processing & Training Configuration
We use `ImageDataGenerator` to load and augment our training set, which helps prevent overfitting and improves generalization. Then, we train the MobileNetV2 model using EarlyStopping and Learning Rate Reduction on Plateau.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Setup Data Generators with Augmentation for the train set
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2 # 20% validation split
)

test_datagen = ImageDataGenerator(rescale=1./255)

# Load data - Assuming data is in 'cell_images' organized into 'Parasitized' and 'Uninfected' subfolders
train_generator = train_datagen.flow_from_directory(
    cell_images_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True
)

val_generator = train_datagen.flow_from_directory(
    cell_images_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

# Callbacks for better training performance
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]

# Initial Training (Phase 1: training only top layers)
history_initial = mobilenet_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks
)

### Fine-Tuning Phase (Experimenting with Unfreezing Layers)
To extract more specific features from the malaria dataset, we can unfreeze some of the top layers of the MobileNetV2 base model and train the entire network end-to-end with a very low learning rate.

In [ ]:
# Unfreeze the base model
base_model.trainable = True

# We want to fine-tune from this layer onwards
fine_tune_at = 100

# Freeze all the layers before the `fine_tune_at` layer
for layer in base_model.layers[:fine_tune_at]:
  layer.trainable = False

# Recompile the model with a lower learning rate
mobilenet_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE / 10),
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')])

# Continue Training (Phase 2: fine-tuning)
FINE_TUNE_EPOCHS = 5
total_epochs = EPOCHS + FINE_TUNE_EPOCHS

history_fine = mobilenet_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=total_epochs,
    initial_epoch=history_initial.epoch[-1],
    callbacks=callbacks
)

### 7 Systematic Experiments Table
To satisfy the requirement of rigorous experimentation, we will run **7 distinct experiments** systematically varying hyperparameters such as Learning Rate, Dropout Rate, and Fine-tuning depth (`fine_tune_at`). We collect the Accuracy, Precision, Recall, and F1-Score for each run to populate a comprehensive benchmark table.

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define 7 distinct experimental setups
experiments = [
    {"exp": 1, "desc": "Baseline Transfer (lr=1e-4, drop=0.5, freeze_all=True)", "lr": 1e-4, "drop": 0.5, "fine_tune_at": 155},
    {"exp": 2, "desc": "High Dropout (drop=0.7) to check overfitting", "lr": 1e-4, "drop": 0.7, "fine_tune_at": 155},
    {"exp": 3, "desc": "Lower Learning Rate (lr=1e-5)", "lr": 1e-5, "drop": 0.5, "fine_tune_at": 155},
    {"exp": 4, "desc": "Fine-Tune Top 10 layers (fine_tune_at=145)", "lr": 1e-4, "drop": 0.5, "fine_tune_at": 145},
    {"exp": 5, "desc": "Fine-Tune Top 30 layers (fine_tune_at=125) lr=1e-5", "lr": 1e-5, "drop": 0.5, "fine_tune_at": 125},
    {"exp": 6, "desc": "Fine-Tune Top 50 layers (fine_tune_at=105)", "lr": 1e-5, "drop": 0.5, "fine_tune_at": 105},
    {"exp": 7, "desc": "Aggressive Fine-Tune (fine_tune_at=80) lr=1e-5", "lr": 1e-5, "drop": 0.5, "fine_tune_at": 80},
]

results_data = []

# This block demonstrates the experimental loop. (Note: Kept to 2 epochs per phase for brevity; increase for true training)
for config in experiments:
    print(f"--- Running Experiment {config['exp']}: {config['desc']} ---")

    # 1. Rebuild base model
    base = MobileNetV2(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,))
    base.trainable = True

    # Freeze layers up to configuration
    for layer in base.layers[:config['fine_tune_at']]:
        layer.trainable = False

    # Build Top
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(config['drop'])(x)
    preds = Dense(1, activation='sigmoid')(x)

    exp_model = Model(inputs=base.input, outputs=preds)
    exp_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=config['lr']),
                      loss='binary_crossentropy', metrics=['accuracy'])

    # 2. Train model (Shortened to 2 epochs for demonstration of the 7 experiments framework)
    exp_model.fit(train_generator, validation_data=val_generator, epochs=2, verbose=0)

    # 3. Evaluate
    val_generator.reset()
    predictions = exp_model.predict(val_generator, verbose=0)
    y_pred_classes = (predictions > 0.5).astype(int).flatten()
    y_true_classes = val_generator.classes

    # 4. Record Metrics
    acc = accuracy_score(y_true_classes, y_pred_classes)
    prec = precision_score(y_true_classes, y_pred_classes, zero_division=0)
    rec = recall_score(y_true_classes, y_pred_classes, zero_division=0)
    f1 = f1_score(y_true_classes, y_pred_classes, zero_division=0)

    results_data.append({
        "Experiment": config['exp'],
        "Description": config['desc'],
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4)
    })

# Render Table
results_df = pd.DataFrame(results_data)
results_df.set_index("Experiment", inplace=True)
display(results_df)

### Model Evaluation & Visualization
According to the project requirements, we need to evaluate the model presenting the results (accuracy, precision, recall, and F1-score), plot learning curves, confusion matrix, and ROC/AUC curves.

In [ ]:
# 1. Learning Curves
acc = history_initial.history['accuracy'] + history_fine.history['accuracy']
val_acc = history_initial.history['val_accuracy'] + history_fine.history['val_accuracy']
loss = history_initial.history['loss'] + history_fine.history['loss']
val_loss = history_initial.history['val_loss'] + history_fine.history['val_loss']

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.plot([EPOCHS-1, EPOCHS-1], plt.ylim(), label='Start Fine Tuning')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.plot([EPOCHS-1, EPOCHS-1], plt.ylim(), label='Start Fine Tuning')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

# Predict on validation set
val_generator.reset()
preds = mobilenet_model.predict(val_generator)
y_pred = (preds > 0.5).astype(int).flatten()
y_true = val_generator.classes

# 2. Confusion Matrix & Classification Report
cm = confusion_matrix(y_true, y_pred)
print("Classification Report:\n", classification_report(y_true, y_pred, target_names=['Parasitized', 'Uninfected']))

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Parasitized', 'Uninfected'], yticklabels=['Parasitized', 'Uninfected'])
plt.title('Confusion Matrix: MobileNetV2')
plt.ylabel('True Class')
plt.xlabel('Predicted Class')
plt.show()

# 3. ROC and AUC Curve
fpr, tpr, thresholds = roc_curve(y_true, preds)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic - MobileNetV2')
plt.legend(loc="lower right")
plt.show()